In [2]:
import os
import torch
import numpy as np
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer
from datasets import Dataset
from peft import PeftModel, AutoPeftModelForCausalLM
import sys
from transformers import AutoModelForCausalLM

# Add path for Expression class
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../classes')))
from expression import Expression
from dataset import RegressionDataset

# === Reward function ===
def compute_reward(expression_str: str) -> float:
    try:
        expr = Expression(expression_str)
        score = expr.fit_constants(X, y)
        return float(score) if np.isfinite(score) else -1.0
    except Exception as e:
        print(f"Erro ao avaliar expressão: {expression_str} - {e}")
        return -1.0

# === Helper to extract expression ===
def extract_expression(response: str) -> str:
    return response.split("expr: ")[1].split("<|endoftext|>")[0].strip()

# === Load Data ===
reg = RegressionDataset('../data/evaluate/srsd-feynman_easy/train', 'feynman-i.12.1.txt', delimiter=' ')
X, y = reg.get_numpy()

# === Configs ===
BASE_MODEL = "augustocsc/Se124M100KInfPrompt_EOS_Merged"
LORA_REPO = "augustocsc/Se124M100KInfPrompt_EOS_Merged"
TOKENIZER_REPO = LORA_REPO

ppo_config = PPOConfig(
    #model_name=BASE_MODEL,
    learning_rate=1e-5,
    batch_size=16,
    mini_batch_size=4,
    gradient_accumulation_steps=1,
)

model = AutoModelForCausalLMWithValueHead.from_pretrained(BASE_MODEL)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model = model.to(device)
ref_model = ref_model.to(device)

# === PPO Trainer ===
ppo_trainer = PPOTrainer(
    config=ppo_config,
    tokenizer=tokenizer,
    model=model,
    ref_model=ref_model,
    
)

/home/augusto/symbo_repos/seringuela/.seriguela/lib/python3.11/site-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(
Some weights of the model checkpoint at augustocsc/Se124M100KInfPrompt_EOS_Merged were not used when initializing GPT2LMHeadModel: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at augustocsc/Se124M100KInfPrompt_EOS_Merged were not used when

In [ ]:
from tqdm import tqdm

PROMPT = """
vars: x_1, x_2
oper: *, **, +, -, /
cons: C
expr:"""

# === Dummy dataset ===
dummy_dataset = Dataset.from_dict({
    "prompt": [PROMPT] * 100
})


# Get the device of the model
device = next(model.parameters()).device

# === PPO Training Loop ===
# Tokenize the prompt and convert it to tensors
inputs = tokenizer([PROMPT] * ppo_config.batch_size, return_tensors="pt", padding=True)

# Move inputs to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}

# Convert the batch tensor into a list of individual tensors
queries = [inputs["input_ids"][i] for i in range(inputs["input_ids"].size(0))]


for epoch in tqdm(range(10), desc="Training Epochs"):  # adjust as needed
    responses = []
    constants = []
    for i in tqdm(range(ppo_config.batch_size), desc="Batch Progress", leave=False):  # Nested progress bar
        output = model.generate(
            input_ids=inputs["input_ids"][i].unsqueeze(0),
            attention_mask=inputs["attention_mask"][i].unsqueeze(0),
            max_new_tokens=50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            eos_token_id=tokenizer.eos_token_id,       
            pad_token_id=tokenizer.pad_token_id        
        )
        generated = tokenizer.decode(output[0], skip_special_tokens=True)
        response = generated[len(PROMPT):].strip()
        responses.append(response)

    rewards = [compute_reward(r) for r in responses]
    print(f"Epoch {epoch + 1} - Rewards: {rewards}")
    # print each response and its corresponding reward
    for response, reward in zip(responses, rewards):
        print(f"Response: {response} - Reward: {reward}")
    
    break
    # Convert rewards to a list of PyTorch tensors
    rewards = [torch.tensor(reward, dtype=torch.float32, device=device) for reward in rewards]
    
    # Ensure responses are also tokenized and converted to tensors
    responses = [tokenizer(response, return_tensors="pt", padding=True)["input_ids"].squeeze(0).to(device) for response in responses]

    # Pass the tokenized tensors to ppo_trainer.step()
    ppo_trainer.step(queries, responses, rewards)

    # Log top expressions
    top_k = 3
    sorted_responses = sorted(zip(responses, rewards), key=lambda x: -x[1])
    print(f"\nEpoch {epoch + 1} melhores expressões:")
    for i, (expr, score) in enumerate(sorted_responses[:top_k]):
        print(f"{i+1}. {tokenizer.decode(expr, skip_special_tokens=True)} -> R² = {score:.4f}")


Training Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeWarning: invalid value encountered in scalar power
<string>:1: RuntimeW